In [ ]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/hardsharer/')
sys.path.append('/kaggle/input/cmi-competition-code')

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report

import data_utils
from utils import SequenceExtractor
from utils_proto import BinaryPlusGesturePrototypicalNetwork, DynamicMultiHeadPrototypicalNetwork

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
# ============================================================
# CONFIGURATION - CHANGE THIS TO SWITCH MODES
# ============================================================

# Set this to "single" or "multi"
mode = "single"  # "single" for binary+gesture, "multi" for multi-head

single_target = "gesture"  # For single mode: "gesture", "gesture_action", etc.

# For MULTI mode: which heads to predict
multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "gesture"  # For multi mode: what to evaluate F1 on

pipe_name = "extractor"
proto_name = "model"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 5
n_jobs = 1
train_size = 0.6
error_score_constant = np.nan
verbose = 3

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print(f"Mode: {mode}")
print(f"Search mode: {search_mode}")

In [ ]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

In [ ]:
train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)
q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
bad_q = norm.squeeze() == 0
q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
q_wxyz = q_wxyz / norm

q_xyzw = q_wxyz[:, [1, 2, 3, 0]]
euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)
euler_xyz[:, [1, 2]] *= -1.0
q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()
q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]
train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed

print(f"left-handed corrected: {left_handed_mask.sum()} rows")

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
print(f"upside-down corrected: {upside_down_mask.sum()} rows")

train_df = train_df.drop(columns=["handedness"])

In [ ]:
# Create target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

train_sample_df, hold_out_df = data_utils.sample_balanced_split(
    train_df, train_pct=train_size, test_pct=(1 - train_size), random_state=random_state
)

print(f"Train sequences: {train_sample_df['sequence_id'].nunique()}")
print(f"Hold Out sequences: {hold_out_df['sequence_id'].nunique()}")

In [ ]:

# ============================================================
# CONFIGURATION - CHANGE THESE AT THE TOP
# ============================================================

# Model type: "single" or "multi"
mode = "single"  # "single" = binary + gesture, "multi" = multi-head

# For SINGLE mode: uses is_target and gesture (binary + classification)
# No additional config needed - it uses train_df['is_target'] and train_df['gesture']

# For MULTI mode: which heads to predict (columns that exist in dataframe)
multi_heads = ["gesture_action", "orientation", "gesture_position"]  # any combination

# For MULTI mode: what to evaluate F1 on (full gesture string)
primary_target = "bfrb"  # "gesture", "gesture_action", etc.
single_target = "bfrb"  # For single mode: "gesture", "gesture_action", etc.

# Domain combinations (use "|" for multiple, None to disable)
acc_mode = "displacement"                    # "raw", "displacement", "velocity", "jerk", or "raw|displacement|velocity"
rotation_mode = "quaternion"                 # "quaternion", "rot6d", "angular_velocity", or "quaternion|rot6d"  
tof_mode = "sensor_stats|pooled_stats"       # "sensor_stats", "pooled_stats", or "sensor_stats|pooled_stats"
thm_mode = "centered_diff"                   # "raw", "diff", "centered", "centered_diff", or None

# ============================================================
# BUILD EXTRACTOR
# ============================================================

sequence_extractor = SequenceExtractor(
    acc_mode=acc_mode,
    linear_acc_mode="baseline",
    use_acc_magnitude=True,
    use_linear_acc_magnitude=True,
    sampling_rate=20,
    compute_dt=True,
    clip_value=50.0,
    interp_mode="linear",
    use_highpass_fallback=True,
    window_size=7,
    smooth_alpha=0.5,
    standardize="mean_std",
    include_mask=False,
    rotation_mode=rotation_mode,
    fix_quaternion_sign=True,
    tof_mode=tof_mode,
    tof_fill_mode="far_255",
    thm_mode=thm_mode,
)

# ============================================================
# BUILD MODEL BASED ON MODE
# ============================================================

common_params = {
    "backbone_type": "1dcnn",
    "conv_filters": "64-128",
    "kernel_sizes": "5-3",
    "pool_sizes": "none",
    "use_batch_norm": True,
    "spatial_dropout": 0.1,
    "dense_units": "64",
    "dropout": 0.3,
    "embedding_dim": 128,
    "distance": "euclidean",
    "learning_rate": 1e-3,
    "batch_size": 32,
    "epochs": 100,
    "patience": 15,
    "maxlen": 160,
    "verbose": 0,
    "random_state": 42,
}

if mode == "single":
    # BinaryPlusGesturePrototypicalNetwork uses is_target and gesture automatically
    model = BinaryPlusGesturePrototypicalNetwork(
        **common_params,
        gesture_column=single_target,
        n_way=20,
        n_support=5,
        n_query=15,
    )
else:
    model = DynamicMultiHeadPrototypicalNetwork(
        **common_params,
        heads=multi_heads,
        primary_target=primary_target,
        n_way=20,
        n_support=5,
        n_query=15,
    )

pipeline = Pipeline([
    ("extractor", sequence_extractor),
    ("model", model),
])

In [ ]:
if search_mode == "bayesian":
    param_space = {
        # ============================================================
        # SequenceExtractor parameters
        # ============================================================
        f"{pipe_name}__acc_mode": Categorical(["smoothed|velocity|displacement|jerk", "displacement", "velocity", "jerk"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__sampling_rate": Integer(10, 100),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear"]),
        f"{pipe_name}__use_highpass_fallback": Categorical([True]),
        f"{pipe_name}__window_size": Integer(3, 21),
        f"{pipe_name}__smooth_alpha": Categorical([0.5]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__include_mask": Categorical([False]),
        f"{pipe_name}__rotation_mode": Categorical(["quaternion|rot6d|angular_velocity|delta_euler", "quaternion", "rot6d", "angular_velocity"]),
        f"{pipe_name}__fix_quaternion_sign": Categorical([True]),
        f"{pipe_name}__tof_mode": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled_stats|pooled_diff"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["far_255"]),
        f"{pipe_name}__thm_mode": Categorical(["centered_diff"]),
        
        # ============================================================
        # PrototypicalNetwork parameters
        # ============================================================
        f"{proto_name}__backbone_type": Categorical(["1dcnn"]),
        f"{proto_name}__conv_filters": Categorical(["64", "128", "64-128", "128-256"]),
        f"{proto_name}__kernel_sizes": Categorical(["3", "5", "3-3", "5-3", "7-5"]),
        f"{proto_name}__pool_sizes": Categorical(["none", "2", "2-2"]),
        f"{proto_name}__use_batch_norm": Categorical([True]),
        f"{proto_name}__spatial_dropout": Real(0.0, 0.5),
        f"{proto_name}__dense_units": Categorical(["none", "64", "128", "64-32", "128-64"]),
        f"{proto_name}__dropout": Real(0.0, 0.5),
        f"{proto_name}__embedding_dim": Integer(64, 512),
        f"{proto_name}__distance": Categorical(["euclidean", "cosine"]),
        f"{proto_name}__learning_rate": Real(1e-4, 1e-2, prior="log-uniform"),
        f"{proto_name}__batch_size": Categorical([16, 32, 64]),
        f"{proto_name}__epochs": Categorical([100]),
        f"{proto_name}__patience": Categorical([15]),
        f"{proto_name}__maxlen": Integer(80, 300),
        f"{proto_name}__n_way": Categorical([10, 15, 20, 25]),
        f"{proto_name}__n_support": Categorical([3, 5, 10, 15]),
        f"{proto_name}__n_query": Categorical([5, 10, 15, 20]),
    }
else:
    param_space = {
        # ============================================================
        # SequenceExtractor parameters (locked to known good values)
        # ============================================================
        f"{pipe_name}__acc_mode": ["displacement"],
        f"{pipe_name}__linear_acc_mode": ["baseline"],
        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],
        f"{pipe_name}__sampling_rate": [20],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": [7],
        f"{pipe_name}__smooth_alpha": [0.5],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__include_mask": [False],
        f"{pipe_name}__rotation_mode": ["quaternion"],
        f"{pipe_name}__fix_quaternion_sign": [True],
        f"{pipe_name}__tof_mode": ["sensor_stats"],
        f"{pipe_name}__tof_fill_mode": ["far_255"],
        f"{pipe_name}__thm_mode": ["centered_diff"],
        
        # ============================================================
        # PrototypicalNetwork parameters (grid over architecture)
        # ============================================================
        f"{proto_name}__backbone_type": ["1dcnn"],
        f"{proto_name}__conv_filters": ["64"],
        f"{proto_name}__kernel_sizes": ["3"],
        f"{proto_name}__pool_sizes": ["none"],
        f"{proto_name}__use_batch_norm": [True],
        f"{proto_name}__spatial_dropout": [0.0],
        f"{proto_name}__dense_units": ["none"],
        f"{proto_name}__dropout": [0.0],
        f"{proto_name}__embedding_dim": [64],
        f"{proto_name}__distance": ["euclidean"],
        f"{proto_name}__learning_rate": [0.0001],
        f"{proto_name}__batch_size": [16],
        f"{proto_name}__epochs": [100],
        f"{proto_name}__patience": [15],
        f"{proto_name}__maxlen": [120],
        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [3],
        f"{proto_name}__n_query": [5],
    }

In [ ]:
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    # For multi mode, include all heads plus the primary target for lookup
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols:
        cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", primary_target]].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
groups = X_train["sequence_id"]

print(f"MODE: {mode}")
print(f"Single target: {single_target if mode=='single' else 'N/A'}")
print(f"Multi heads: {multi_heads if mode=='multi' else 'N/A'}")
print(f"Primary target: {primary_target if mode=='multi' else 'N/A'}")

print(f"y_train columns: {y_train.columns.tolist()}")

In [ ]:

# Run CV search
cv = GroupKFold(n_splits=n_splits)

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring="f1_macro",
        cv=cv,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant,
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring="f1_macro",
        cv=cv,
        n_jobs=n_jobs,
        verbose=4,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

In [ ]:
# ============================================================
# EVALUATE ON HOLDOUT TEST SET - WORKS FOR BOTH MODES
# ============================================================

# Prepare test labels - USE BFRB COLUMN (it has 'non_bfrb' + gestures)
y_test_true = hold_out_df[["sequence_id", "is_target", "bfrb"]].copy()

# Predict
y_pred = best_model.predict(X_test)

# Binary F1 (target vs non-target)
y_true_binary = y_test_true["is_target"].values.astype(int)
y_pred_binary = (y_pred != "non_bfrb").astype(int)
binary_f1 = f1_score(y_true_binary, y_pred_binary)

# Gesture Macro F1 (only target/BFRB sequences)
target_mask = y_test_true["is_target"] == 1
if target_mask.sum() > 0:
    gesture_f1 = f1_score(
        y_test_true.loc[target_mask, "bfrb"], 
        y_pred[target_mask], 
        average="macro"
    )
else:
    gesture_f1 = 0.0

competition_score = (binary_f1 + gesture_f1) / 2

print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)
print(f"Binary F1 (non_bfrb vs bfrb): {binary_f1:.4f}")
print(f"BFRB Gesture Macro F1: {gesture_f1:.4f}")
print(f"COMPETITION SCORE: {competition_score:.4f}")

if target_mask.sum() > 0:
    print("\n" + "-"*40)
    print("BFRB Gesture Classification Report")
    print("-"*40)
    print(classification_report(
        y_test_true.loc[target_mask, "bfrb"], 
        y_pred[target_mask]
    ))

# Save results
holdout_results = pd.DataFrame({
    "sequence_id": X_test.index.unique(),
    "is_target_true": y_true_binary,
    "is_target_pred": y_pred_binary,
    "bfrb_true": y_test_true["bfrb"].values,
    "bfrb_pred": y_pred,
})
holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_results.to_csv(holdout_results_path, index=False)
print(f"\nHoldout predictions saved to: {holdout_results_path}")

In [ ]:
unique_labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=unique_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=unique_labels, 
            yticklabels=unique_labels)
plt.title(f"Confusion Matrix - {mode} mode\nTarget: {single_target if mode=='single' else primary_target}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = results_dir / f"confusion_matrix_{mode}_{timestamp}.png"
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Confusion matrix saved to: {cm_path}")

print("\n" + "="*60)
print("EXPERIMENT COMPLETE")
print("="*60)